# AML Rule Engine and Model Validation Demo

This notebook demonstrates a synthetic AML monitoring and validation workflow for a public portfolio. It does not contain confidential information, production rule logic, client data, or proprietary thresholds.

## Workflow

Synthetic data -> config-driven normalization -> point-in-time features -> behavioral CRR -> deterministic TM rules -> separately coded rule replication -> alert consolidation -> holdout ML Alert Risk Rating -> validation evidence.

In [ ]:
from aml_mv_demo.config import load_risk_indicator_config, load_rule_config, load_suppression_config
from aml_mv_demo.crr import calculate_behavioral_crr
from aml_mv_demo.data_generator import generate_synthetic_data
from aml_mv_demo.features import build_customer_features
from aml_mv_demo.ml_arr import build_training_frame, evaluate_arr_model, score_alerts, split_training_validation, train_arr_model
from aml_mv_demo.replication import replicate_rule_results
from aml_mv_demo.rules import consolidate_alerts, evaluate_rules
from aml_mv_demo.validation import arr_monitoring_summary, config_coverage_summary, data_quality_summary, exclusion_summary, reconcile_rule_results

AS_OF = "2026-03-31T23:59:59Z"
rule_config = load_rule_config()
suppression_config = load_suppression_config()
risk_indicator_config = load_risk_indicator_config()

In [ ]:
customers, accounts, transactions, outcomes = generate_synthetic_data(seed=42)
print(customers.shape, accounts.shape, transactions.shape, outcomes.shape)
customers.head()

In [ ]:
dq = data_quality_summary(customers, accounts, transactions, risk_indicator_config=risk_indicator_config)
exclusions = exclusion_summary(transactions, AS_OF, risk_indicator_config)
display(dq)
display(exclusions.head())

In [ ]:
features = build_customer_features(transactions, as_of_timestamp=AS_OF, risk_indicator_config=risk_indicator_config)
behavioral_crr = calculate_behavioral_crr(
    customers,
    features.merge(customers[["customer_id", "expected_monthly_volume"]], on="customer_id", how="left"),
)
behavioral_crr.head()

In [ ]:
rule_results = evaluate_rules(
    customers,
    transactions,
    behavioral_crr,
    as_of_timestamp=AS_OF,
    rule_config=rule_config,
    suppression_config=suppression_config,
    risk_indicator_config=risk_indicator_config,
)
alerts = consolidate_alerts(rule_results)
config_coverage = config_coverage_summary(rule_results, rule_config)
display(rule_results.groupby(["rule_id", "triggered", "suppression_applied", "alert_generated"]).size().reset_index(name="count"))
display(config_coverage)

In [ ]:
independent_rule_results = replicate_rule_results(
    customers,
    transactions,
    behavioral_crr,
    as_of_timestamp=AS_OF,
    rule_config=rule_config,
    suppression_config=suppression_config,
    risk_indicator_config=risk_indicator_config,
)
reconciliation = reconcile_rule_results(rule_results, independent_rule_results)
reconciliation["reconciliation_status"].value_counts()

In [ ]:
model_frame = build_training_frame(alerts, behavioral_crr, features, outcomes)
development_frame, validation_frame = split_training_validation(model_frame)
model = train_arr_model(development_frame)
scored_alerts = score_alerts(model, validation_frame)
display(scored_alerts[["alert_id", "customer_id", "typology_family", "ml_arr_score", "ml_arr_band", "ml_reason_codes"]].head(10))
display(evaluate_arr_model(scored_alerts))
display(arr_monitoring_summary(scored_alerts))

## Interpretation

This walkthrough now separates primary rule execution from independently coded replication, uses externalized configuration for rules and risk indicators, reports data-quality and exclusion controls, and scores ML ARR on a holdout validation population. The outputs remain synthetic and should not be interpreted as production detection performance or regulatory conclusions.